# OBELIS dataset

**Source**: NOW GmbH, Nationale Leitstelle Ladeinfrastruktur, Deutschland, https://nationale-leitstelle.de/verstehen/   
**Link**: https://mobilithek.info/offers/714073450865197056

**Beschreibung**:  
Ladevorgänge, die im Rahmen der Halbjahresberichte zu den geförderten Ladepunkten über OBELIS übermittelt worden sind. Enthalten sind aktuell Ladevorgänge bis einschließlich des ersten Halbjahres 2024. Die Meldung der Ladevorgänge für das zweite Halbjahr 2024 findet aktuell noch statt. Diese werden beim nächsten Update enthalten sein.  
Um wegen des berechtigten Geheimhaltungsinteresses der Betreiber die Zuordnung der Ladevorgangsdaten zu den Ladepunkt- und Ladestationsdaten zu verhindern, werden die Ladepunkt IDs sowie die Ladestation IDs in der Ladevorgangstabelle über ein mit Zufallszahlen befülltes dictionary mit neuen IDs überschrieben. Um zu verdeutlichen, dass es sich nicht um die originalen IDs sondern um durchmischte Zufallszahlen handelt, werden zusätzlich die Suffixe „_shuffled“ angehangen. Auf diese Weise ist sichergestellt, dass Ladevorgänge, die an gleichen Ladepunkten stattfanden, weiterhin gleichen Ladepunkten zugeordnet werden. Durch das Überschreiben der IDs mit Zufallszahlen werden die Ladevorgänge aber zufälligen Ladepunkten zugeordnet und nicht den Ladepunkten, an denen sie tatsächlich stattfanden. Es können folglich keine Rückschlüsse mehr auf Betriebsgeheimnisse gezogen werden.   
Damit zumindest eingeschränkte räumliche Analysen zur Auslastung angefertigt werden können, werden an die Ladevorgänge vor dem „Shuffle“ der Ladestation-IDs Informationen zum Bundesland und zur Lage von der tatsächlichen Ladestation beigefügt. Genauere Informationen zur PLZ oder Ort werden wegen des Geheimhaltungsinteresses nicht beigefügt.

In [5]:
import pandas as pd

def print_metadata(current_df):
    print(f'Das Dataframe hat {len(current_df)} Einträge')
    print(f'Von {current_df["beginn"].min()} bis {current_df["ende"].max()}')
    print(f'{current_df["ls_id"].nunique()} Ladestationen mit insg. {current_df["lp_id"].nunique()} Ladepunkten')

In [2]:
df = pd.read_csv("df_lv.csv", delimiter=";")
print_metadata(df)
df.head()

Das Dataframe hat 19945103 Einträge
Von 2018-07-01 00:18:00 bis 2024-01-01 17:22:08
13787 Ladestationen mit insg. 27980 Ladepunkten


,lv_id,beginn,ende,dauer_sekunden,energie_wh,lp_id,ls_id,bundesland,lage,maxladeleistunginkilowatt
0,30286855,2023-08-27 17:01:55,2023-08-27 18:28:04,5169.0,15717.0,18409_shuffled,965_shuffled,Nordrhein-Westfalen,Kundenparkplatz,22.0
1,30286853,2023-09-01 19:20:49,2023-09-01 19:44:38,1429.0,1377.0,18409_shuffled,965_shuffled,Nordrhein-Westfalen,Kundenparkplatz,22.0
2,30286851,2023-09-09 16:31:12,2023-09-09 19:53:39,12147.0,27179.0,18409_shuffled,965_shuffled,Nordrhein-Westfalen,Kundenparkplatz,22.0
3,30286848,2023-09-15 08:44:13,2023-09-15 09:34:20,3007.0,13850.0,18409_shuffled,965_shuffled,Nordrhein-Westfalen,Kundenparkplatz,22.0
4,30286847,2023-09-16 17:22:31,2023-09-16 17:51:03,1712.0,1568.0,18409_shuffled,965_shuffled,Nordrhein-Westfalen,Kundenparkplatz,22.0


In [ ]:
# Nur Autobahn Ladesäulen
df_autobahn = df[df["lage"] == "Tankstelle an einer Bundesautobahn"]
print_metadata(df_autobahn)

Das Dataframe hat 668365 Einträge
Von 2018-07-01 00:39:00 bis 2023-12-31 23:36:46
140 Ladestationen mit insg. 373 Ladepunkten


In [ ]:
# Am häufigsten frequentierte Station?
value_count = df_autobahn["ls_id"].value_counts()
value_count

ls_id
1590_shuffled     44687
12651_shuffled    35704
11761_shuffled    31671
12209_shuffled    27535
52_shuffled       24958
                  ...  
13629_shuffled     1002
4018_shuffled       838
4782_shuffled       707
6674_shuffled       658
6080_shuffled       157
Name: count, Length: 140, dtype: int64

In [15]:
df_ls1590 = df_autobahn[df_autobahn["ls_id"] == "1590_shuffled"]
print_metadata(df_ls1590)

Das Dataframe hat 44687 Einträge
Von 2020-12-30 17:31:46 bis 2023-09-20 00:47:52
1 Ladestationen mit insg. 4 Ladepunkten


In [ ]:
# Am häufigsten frequentierte Säule?
value_count = df_autobahn["lp_id"].value_counts()
value_count

lp_id
6804_shuffled     12044
6973_shuffled     11552
16564_shuffled    10758
6326_shuffled     10333
24193_shuffled     9693
                  ...  
24020_shuffled      113
20560_shuffled       88
14933_shuffled       84
22862_shuffled       45
25346_shuffled        3
Name: count, Length: 373, dtype: int64

In [31]:
df_lp6804 = df_autobahn[df_autobahn["lp_id"] == "6804_shuffled"]
print_metadata(df_lp6804)

Das Dataframe hat 12044 Einträge
Von 2021-01-02 11:31:36 bis 2023-09-20 00:47:52
1 Ladestationen mit insg. 1 Ladepunkten


In [19]:
import pandas as pd

def find_busiest_two_hour_window(df):
    # Ensure the 'beginn' column is in datetime format
    df['beginn'] = pd.to_datetime(df['beginn'])

    # Group by hour and minute, then count
    counts = df.groupby(df['beginn'].dt.floor('1T')).size()

    # Calculate rolling 2-hour sum
    rolling_sum = counts.rolling('2H').sum()

    # Find the busiest window
    busiest_time = rolling_sum.idxmax()
    max_count = int(rolling_sum.max())

    return busiest_time - pd.Timedelta(hours=2), busiest_time, max_count



In [20]:
start_time, end_time, count = find_busiest_two_hour_window(df_ls1590)

print(f"The busiest two-hour window is from {start_time} to {end_time}")
print(f"Number of entries: {count}")

The busiest two-hour window is from 2023-04-29 17:11:00 to 2023-04-29 19:11:00
Number of entries: 20


/tmp/ipykernel_4571/4290575426.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['beginn'] = pd.to_datetime(df['beginn'])
/tmp/ipykernel_4571/4290575426.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  counts = df.groupby(df['beginn'].dt.floor('1T')).size()
/tmp/ipykernel_4571/4290575426.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rolling_sum = counts.rolling('2H').sum()


In [23]:
# filter for the day with the busiest two hours
# df_ls1590['beginn'] = pd.to_datetime(df_ls1590['beginn'])
df_ls1590_busiest = df_ls1590[df_ls1590['beginn'].dt.date == pd.to_datetime('2023-04-29').date()]
print_metadata(df_ls1590_busiest)

Das Dataframe hat 86 Einträge
Von 2023-04-29 00:28:11 bis 2023-04-29 23:05:17
1 Ladestationen mit insg. 4 Ladepunkten


In [29]:
df_ls1590_busiest.sort_values(by="beginn").tail(30)

,lv_id,beginn,ende,dauer_sekunden,energie_wh,lp_id,ls_id,bundesland,lage,maxladeleistunginkilowatt
16647961,20606724,2023-04-29 16:41:12,2023-04-29 17:23:02,2510.0,36662.0,6326_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14374025,20602797,2023-04-29 16:46:14,2023-04-29 17:01:06,892.0,20424.0,16564_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14368498,20598926,2023-04-29 16:47:20,2023-04-29 17:17:35,1815.0,44299.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
3321960,20610776,2023-04-29 16:50:56,2023-04-29 17:14:37,1421.0,40902.0,6973_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14368499,20598927,2023-04-29 17:21:52,2023-04-29 17:42:14,1222.0,32500.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14374026,20602798,2023-04-29 17:42:19,2023-04-29 18:19:33,2234.0,31146.0,16564_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
16647962,20606725,2023-04-29 17:43:57,2023-04-29 18:03:39,1182.0,36866.0,6326_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14368500,20598928,2023-04-29 17:53:38,2023-04-29 18:19:15,1537.0,36014.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
3321961,20610778,2023-04-29 17:54:46,2023-04-29 18:31:01,2175.0,50733.0,6973_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
16647963,20606726,2023-04-29 18:09:57,2023-04-29 18:32:39,1362.0,33980.0,6326_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0


In [32]:
start_time, end_time, count = find_busiest_two_hour_window(df_lp6804)

print(f"The busiest two-hour window is from {start_time} to {end_time}")
print(f"Number of entries: {count}")

The busiest two-hour window is from 2023-03-05 10:50:00 to 2023-03-05 12:50:00
Number of entries: 8


/tmp/ipykernel_4571/4290575426.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['beginn'] = pd.to_datetime(df['beginn'])
/tmp/ipykernel_4571/4290575426.py:8: FutureWarning: 'T' is deprecated and will be removed in a future version, please use 'min' instead.
  counts = df.groupby(df['beginn'].dt.floor('1T')).size()
/tmp/ipykernel_4571/4290575426.py:11: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  rolling_sum = counts.rolling('2H').sum()


In [33]:
# filter df for the day with the busiest two hours
df_lp6804_busiest = df_lp6804[df_lp6804['beginn'].dt.date == pd.to_datetime('2023-03-05').date()]
print_metadata(df_lp6804_busiest)

df_lp6804_busiest.sort_values(by="beginn").head(30)

Das Dataframe hat 26 Einträge
Von 2023-03-05 00:44:55 bis 2023-03-05 22:13:43
1 Ladestationen mit insg. 1 Ladepunkten


,lv_id,beginn,ende,dauer_sekunden,energie_wh,lp_id,ls_id,bundesland,lage,maxladeleistunginkilowatt
14367489,20597475,2023-03-05 00:44:55,2023-03-05 00:55:57,662.0,22577.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367490,20597476,2023-03-05 08:47:08,2023-03-05 08:51:26,258.0,6249.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367491,20597477,2023-03-05 09:45:08,2023-03-05 10:18:45,2017.0,86892.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367492,20597478,2023-03-05 10:21:21,2023-03-05 10:48:40,1639.0,41849.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367493,20597479,2023-03-05 10:52:40,2023-03-05 10:58:54,374.0,5249.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367494,20597480,2023-03-05 11:03:04,2023-03-05 11:04:52,108.0,3394.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367495,20597482,2023-03-05 11:18:14,2023-03-05 11:30:56,762.0,14511.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367496,20597485,2023-03-05 11:36:47,2023-03-05 11:52:28,941.0,13489.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367497,20597486,2023-03-05 11:57:22,2023-03-05 12:24:24,1622.0,35591.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0
14367498,20597487,2023-03-05 12:27:59,2023-03-05 12:35:22,443.0,7215.0,6804_shuffled,1590_shuffled,Hessen,Tankstelle an einer Bundesautobahn,350.0


---
## Testen des Obelis_Data_Provider

In [12]:
csv_path="df_lv.csv"
df = pd.read_csv(csv_path, delimiter=";", parse_dates=["beginn", "ende"])

In [17]:
lp_ids = ['6804_shuffled', '6973_shuffled', '16564_shuffled', '6326_shuffled']
filter_date = '2023-03-05'
df_filtered = df[df["lp_id"].isin(lp_ids)]
df_filtered = df_filtered[df_filtered["beginn"].dt.date == pd.to_datetime(filter_date).date()]

In [18]:
print_metadata(df_filtered)

Das Dataframe hat 89 Einträge
Von 2023-03-05 00:44:55 bis 2023-03-05 23:36:24
1 Ladestationen mit insg. 4 Ladepunkten


In [ ]:
dataframe = df_filtered

def __match_cs(lp_id):
    cs_matching_dict = {
        lp_ids[0]: "cs_0",
        lp_ids[1]: "cs_1",
        lp_ids[2]: "cs_2",
        lp_ids[3]: "cs_3"
    }
    return cs_matching_dict[lp_id]

def __convert_to_depart_time(charge_begin_datetime):
    hours = charge_begin_datetime.hour
    minutes = charge_begin_datetime.minute
    seconds = charge_begin_datetime.second
    return (hours * 3600) + (minutes * 60) + seconds

def __convert_data_from_csv_row(lp_id, charge_begin_datetime, charge_duration):
    cs_id = __match_cs(lp_id)
    charge_begin_seconds = __convert_to_depart_time(charge_begin_datetime)
    result_dict = {
        "cs_id": cs_id,
        "charge_begin_seconds": charge_begin_seconds,
        "charge_duration": charge_duration
    }
    return result_dict

vehicle_data = []
for row in dataframe.itertuples(index=False):
    result_dict = __convert_data_from_csv_row(row.lp_id, row.beginn, row.dauer_sekunden)
    vehicle_data.append(result_dict)


[{'cs_id': 'cs_3', 'charge_begin_seconds': 7692, 'charge_duration': 645.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 29560, 'charge_duration': 1871.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 35780, 'charge_duration': 1923.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 38409, 'charge_duration': 1425.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 41013, 'charge_duration': 1246.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 42903, 'charge_duration': 666.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 43944, 'charge_duration': 971.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 45113, 'charge_duration': 2127.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 47509, 'charge_duration': 1680.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 49403, 'charge_duration': 1071.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 51005, 'charge_duration': 2866.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 54020, 'charge_duration': 1271.0}, {'cs_id': 'cs_3', 'charge_begin_seconds': 56059, 'charge_duration': 1905.0}, {'

In [20]:
for entry in vehicle_data:
    print(entry)

{'cs_id': 'cs_3', 'charge_begin_seconds': 7692, 'charge_duration': 645.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 29560, 'charge_duration': 1871.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 35780, 'charge_duration': 1923.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 38409, 'charge_duration': 1425.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 41013, 'charge_duration': 1246.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 42903, 'charge_duration': 666.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 43944, 'charge_duration': 971.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 45113, 'charge_duration': 2127.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 47509, 'charge_duration': 1680.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 49403, 'charge_duration': 1071.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 51005, 'charge_duration': 2866.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 54020, 'charge_duration': 1271.0}
{'cs_id': 'cs_3', 'charge_begin_seconds': 56059, 'charge_duration': 1905.0}
{'cs_id': 'cs_3'

In [ ]:
# Convert the csv file to feather-format

import pandas as pd

df = pd.read_csv('df_lv.csv', delimiter=";", parse_dates=["beginn", "ende"])
df.to_feather('df_lv.feather')
